# Y1 486期完整序列 TCN

使用 `X[t-485:t+1]` 预测 `Y1[t]`，动态构造完整序列并生成独立预测文件。

## 环境

In [1]:
import importlib.util
import subprocess
import sys

required_packages = {
    "numpy": "numpy",
    "pandas": "pandas",
    "zstd": "zstd",
    "torch": "torch",
}
missing_packages = [
    package_name
    for module_name, package_name in required_packages.items()
    if importlib.util.find_spec(module_name) is None
]
if missing_packages:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", *missing_packages]
    )
    importlib.invalidate_caches()

print("依赖检查完成")

依赖检查完成


## 参数

In [2]:
import math
import pickle
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import zstd

DATA_PATH = Path("../../data.z")
OUTPUT_DIR = Path("outputs")
MODEL_PATH = OUTPUT_DIR / "tcn_y1_model.pt"
PREDICTION_PATH = OUTPUT_DIR / "tcn_y1.npy"

SEED = 42
WINDOW_SIZE = 486
FEATURE_COUNT = 99
MODEL_INPUT_CHANNELS = 100
HIDDEN_CHANNELS = 32
DILATIONS = (1, 2, 4, 8, 16, 32, 64, 128)
DROPOUT = 0.1
TRAIN_TIME_BINS = 64
TIMES_PER_BIN = 8
STOCKS_PER_TIME = 200
VALIDATION_TIME_COUNT = 81
VALIDATION_STOCKS_PER_TIME = 512
BATCH_SIZE = 128
MAX_EPOCHS = 8
EARLY_STOPPING_PATIENCE = 2
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
GRADIENT_CLIP = 1.0
STATISTICS_TIME_CHUNK = 16
VARIANCE_FLOOR = 1e-6
BASELINE_VALIDATION_RANK_IC = 0.089678

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = DEVICE.type == "cuda"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(
    {
        "device": str(DEVICE),
        "amp": AMP_ENABLED,
        "window_size": WINDOW_SIZE,
        "batch_size": BATCH_SIZE,
    }
)

{'device': 'cuda', 'amp': True, 'window_size': 486, 'batch_size': 128}


## 数据

In [3]:
def read_zstd_pickle(path):
    compressed_bytes = pd.read_pickle(path)
    decompressed_bytes = zstd.loads(compressed_bytes)
    return pickle.loads(decompressed_bytes)


data = read_zstd_pickle(DATA_PATH)

T, STOCK_COUNT, observed_feature_count = data["num_x"].shape
train_start_idx = int(data["train_start_idx"])
valid_start_idx = int(data["valid_start_idx"])
test_start_idx = int(data["test_start_idx"])

assert observed_feature_count == FEATURE_COUNT
assert train_start_idx == 486
assert valid_start_idx == 2918
assert test_start_idx == 3161
assert T == 3603
assert data["mask_x"].shape == (T, STOCK_COUNT)
assert data["mask_y"].shape == (T, STOCK_COUNT)
assert data["y1"].shape == (T, STOCK_COUNT)

data_summary = pd.Series(
    {
        "time_points": T,
        "stocks": STOCK_COUNT,
        "features": FEATURE_COUNT,
        "train_targets": valid_start_idx - train_start_idx,
        "validation_targets": test_start_idx - valid_start_idx,
        "test_targets": T - test_start_idx,
    },
    name="data",
)
display(data_summary)

time_points           3603
stocks                5282
features                99
train_targets         2432
validation_targets     243
test_targets           442
Name: data, dtype: int64

## 训练期标准化

In [4]:
statistics_stop_idx = valid_start_idx
feature_sum = np.zeros(FEATURE_COUNT, dtype=np.float64)
feature_squared_sum = np.zeros(FEATURE_COUNT, dtype=np.float64)
statistics_sample_count = 0

for chunk_start in range(
    0,
    statistics_stop_idx,
    STATISTICS_TIME_CHUNK,
):
    chunk_stop = min(
        chunk_start + STATISTICS_TIME_CHUNK,
        statistics_stop_idx,
    )
    feature_block = data["num_x"][chunk_start:chunk_stop]
    valid_block = data["mask_x"][chunk_start:chunk_stop]
    valid_features = feature_block[valid_block]
    feature_sum += valid_features.sum(axis=0, dtype=np.float64)
    feature_squared_sum += np.square(
        valid_features,
        dtype=np.float64,
    ).sum(axis=0)
    statistics_sample_count += valid_features.shape[0]

assert statistics_sample_count > 0
assert statistics_stop_idx == valid_start_idx
assert statistics_stop_idx <= test_start_idx

feature_mean = feature_sum / statistics_sample_count
feature_variance = (
    feature_squared_sum / statistics_sample_count
    - np.square(feature_mean)
)
feature_variance = np.maximum(feature_variance, VARIANCE_FLOOR)
feature_std = np.sqrt(feature_variance)
feature_mean = feature_mean.astype(np.float32)
feature_std = feature_std.astype(np.float32)

assert feature_mean.shape == (FEATURE_COUNT,)
assert feature_std.shape == (FEATURE_COUNT,)
assert np.all(np.isfinite(feature_mean))
assert np.all(np.isfinite(feature_std))
assert np.all(feature_std > 0)

display(
    pd.Series(
        {
            "statistics_stop_exclusive": statistics_stop_idx,
            "valid_feature_rows": statistics_sample_count,
            "minimum_std": float(feature_std.min()),
            "maximum_std": float(feature_std.max()),
        },
        name="standardization",
    )
)

statistics_stop_exclusive    2.918000e+03
valid_feature_rows           8.220552e+06
minimum_std                  9.235690e-01
maximum_std                  4.853284e+00
Name: standardization, dtype: float64

## 动态滑窗与缺失填充

In [5]:
def window_bounds(time_idx):
    window_start = time_idx - WINDOW_SIZE + 1
    window_stop = time_idx + 1
    if window_start < 0:
        raise ValueError(time_idx)
    if window_stop - window_start != WINDOW_SIZE:
        raise AssertionError((window_start, window_stop))
    return window_start, window_stop


def prepare_window_arrays(
    feature_window,
    valid_window,
    normalization_mean,
    normalization_std,
):
    valid_window = np.asarray(valid_window, dtype=bool)
    feature_window = np.asarray(feature_window, dtype=np.float32)
    valid_counts = valid_window.sum(axis=0, dtype=np.int32)
    nonempty = valid_counts > 0
    denominators = np.maximum(valid_counts, 1).astype(np.float32)[:, None]
    feature_sums = np.sum(
        feature_window,
        axis=0,
        where=valid_window[:, :, None],
        dtype=np.float64,
    ).astype(np.float32)
    window_means = feature_sums / denominators
    window_means[~nonempty] = normalization_mean
    filled_window = np.where(
        valid_window[:, :, None],
        feature_window,
        window_means[None, :, :],
    )
    standardized_window = (
        filled_window - normalization_mean[None, None, :]
    ) / normalization_std[None, None, :]
    mask_channel = valid_window[:, :, None].astype(np.float32)
    model_inputs = np.concatenate(
        [standardized_window, mask_channel],
        axis=2,
    ).transpose(1, 2, 0)
    coverage = (
        valid_counts.astype(np.float32) / feature_window.shape[0]
    )[:, None]
    return (
        np.ascontiguousarray(model_inputs, dtype=np.float32),
        np.ascontiguousarray(coverage, dtype=np.float32),
        nonempty,
    )


def build_model_batch(time_idx, stock_indices):
    stock_indices = np.asarray(stock_indices, dtype=np.int64)
    window_start, window_stop = window_bounds(time_idx)
    feature_window = data["num_x"][
        window_start:window_stop,
        stock_indices,
        :,
    ]
    valid_window = data["mask_x"][
        window_start:window_stop,
        stock_indices,
    ]
    return prepare_window_arrays(
        feature_window,
        valid_window,
        feature_mean,
        feature_std,
    )


assert window_bounds(486) == (1, 487)

synthetic_features = np.array(
    [
        [[1.0, 2.0], [0.0, 0.0]],
        [[0.0, 0.0], [0.0, 0.0]],
        [[5.0, 6.0], [0.0, 0.0]],
        [[0.0, 0.0], [0.0, 0.0]],
    ],
    dtype=np.float32,
)
synthetic_mask = np.array(
    [
        [True, False],
        [False, False],
        [True, False],
        [False, False],
    ]
)
synthetic_mean = np.array([10.0, 20.0], dtype=np.float32)
synthetic_std = np.array([2.0, 5.0], dtype=np.float32)
synthetic_inputs, synthetic_coverage, synthetic_nonempty = (
    prepare_window_arrays(
        synthetic_features,
        synthetic_mask,
        synthetic_mean,
        synthetic_std,
    )
)
synthetic_reconstructed = (
    synthetic_inputs[:, :2, :].transpose(2, 0, 1)
    * synthetic_std[None, None, :]
    + synthetic_mean[None, None, :]
)

assert np.allclose(synthetic_reconstructed[1, 0], [3.0, 4.0])
assert np.allclose(synthetic_reconstructed[3, 0], [3.0, 4.0])
assert np.allclose(synthetic_reconstructed[:, 1], [10.0, 20.0])
assert np.allclose(synthetic_inputs[0, 2], [1.0, 0.0, 1.0, 0.0])
assert np.allclose(synthetic_inputs[1, 2], 0.0)
assert np.allclose(synthetic_coverage[:, 0], [0.5, 0.0])
assert synthetic_nonempty.tolist() == [True, False]

print("滑窗、均值填充、mask和覆盖率检查通过")

滑窗、均值填充、mask和覆盖率检查通过


## TCN

In [6]:
class CausalResidualBlock(nn.Module):
    def __init__(self, channels, dilation, dropout):
        super().__init__()
        self.left_padding = nn.ConstantPad1d(
            (2 * dilation, 0),
            0.0,
        )
        self.convolution = nn.Conv1d(
            channels,
            channels,
            kernel_size=3,
            dilation=dilation,
        )
        self.normalization = nn.GroupNorm(4, channels)
        self.activation = nn.GELU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, inputs):
        hidden = self.left_padding(inputs)
        hidden = self.convolution(hidden)
        hidden = self.normalization(hidden)
        hidden = self.activation(hidden)
        hidden = self.dropout(hidden)
        return inputs + hidden


class WindowTCN(nn.Module):
    def __init__(
        self,
        input_channels,
        hidden_channels,
        dilations,
        dropout,
    ):
        super().__init__()
        self.projection = nn.Conv1d(
            input_channels,
            hidden_channels,
            kernel_size=1,
        )
        self.projection_activation = nn.GELU()
        self.blocks = nn.ModuleList(
            [
                CausalResidualBlock(
                    hidden_channels,
                    dilation,
                    dropout,
                )
                for dilation in dilations
            ]
        )
        self.head = nn.Sequential(
            nn.Linear(hidden_channels + 1, hidden_channels),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_channels, 1),
            nn.Sigmoid(),
        )

    def forward(self, sequence_inputs, coverage):
        hidden = self.projection_activation(
            self.projection(sequence_inputs)
        )
        for block in self.blocks:
            hidden = block(hidden)
        final_hidden = hidden[:, :, -1]
        combined = torch.cat([final_hidden, coverage], dim=1)
        return self.head(combined)


model = WindowTCN(
    MODEL_INPUT_CHANNELS,
    HIDDEN_CHANNELS,
    DILATIONS,
    DROPOUT,
).to(DEVICE)

receptive_field = 1 + 2 * sum(DILATIONS)
assert receptive_field == 511
assert receptive_field >= WINDOW_SIZE

with torch.no_grad():
    test_sequence = torch.zeros(
        2,
        MODEL_INPUT_CHANNELS,
        WINDOW_SIZE,
        device=DEVICE,
    )
    test_coverage = torch.tensor(
        [[1.0], [0.5]],
        dtype=torch.float32,
        device=DEVICE,
    )
    test_output = model(test_sequence, test_coverage)

assert test_output.shape == (2, 1)
assert torch.all(torch.isfinite(test_output))

print(
    {
        "parameters": sum(
            parameter.numel()
            for parameter in model.parameters()
        ),
        "receptive_field": receptive_field,
    }
)

{'parameters': 29697, 'receptive_field': 511}


## 抽样与验证

In [7]:
def history_available_mask(time_idx):
    window_start, window_stop = window_bounds(time_idx)
    return data["mask_x"][window_start:window_stop].any(axis=0)


def labeled_stock_indices(time_idx):
    label_available = (
        data["mask_y"][time_idx]
        & np.isfinite(data["y1"][time_idx])
    )
    return np.flatnonzero(
        label_available & history_available_mask(time_idx)
    )


def sample_training_times(rng):
    time_bins = np.array_split(
        np.arange(
            train_start_idx,
            valid_start_idx,
            dtype=np.int64,
        ),
        TRAIN_TIME_BINS,
    )
    selected_times = []
    for time_bin in time_bins:
        sample_count = min(TIMES_PER_BIN, time_bin.size)
        selected_times.extend(
            rng.choice(
                time_bin,
                size=sample_count,
                replace=False,
            ).tolist()
        )
    selected_times = np.asarray(selected_times, dtype=np.int64)
    rng.shuffle(selected_times)
    return selected_times


def rank_values(values):
    return (
        pd.Series(values)
        .rank(method="average")
        .to_numpy(dtype=np.float64)
    )


def calculate_rank_ic(predictions, labels):
    usable = np.isfinite(predictions) & np.isfinite(labels)
    if np.count_nonzero(usable) < 2:
        return np.nan
    prediction_ranks = rank_values(predictions[usable])
    label_ranks = rank_values(labels[usable])
    if prediction_ranks.std() == 0 or label_ranks.std() == 0:
        return np.nan
    return float(
        np.corrcoef(
            prediction_ranks,
            label_ranks,
        )[0, 1]
    )


def predict_stock_indices(time_idx, stock_indices):
    stock_indices = np.asarray(stock_indices, dtype=np.int64)
    predictions = np.full(
        stock_indices.size,
        0.5,
        dtype=np.float32,
    )
    model.eval()
    for batch_start in range(
        0,
        stock_indices.size,
        BATCH_SIZE,
    ):
        batch_stop = min(
            batch_start + BATCH_SIZE,
            stock_indices.size,
        )
        batch_stocks = stock_indices[batch_start:batch_stop]
        batch_inputs, batch_coverage, batch_nonempty = (
            build_model_batch(time_idx, batch_stocks)
        )
        if not np.any(batch_nonempty):
            continue
        usable_positions = np.flatnonzero(batch_nonempty)
        input_tensor = torch.from_numpy(
            batch_inputs[usable_positions]
        ).to(DEVICE, non_blocking=True)
        coverage_tensor = torch.from_numpy(
            batch_coverage[usable_positions]
        ).to(DEVICE, non_blocking=True)
        with torch.inference_mode():
            with torch.autocast(
                device_type=DEVICE.type,
                dtype=torch.float16,
                enabled=AMP_ENABLED,
            ):
                batch_predictions = model(
                    input_tensor,
                    coverage_tensor,
                )
        predictions[
            batch_start + usable_positions
        ] = (
            batch_predictions
            .squeeze(1)
            .float()
            .cpu()
            .numpy()
        )
    return predictions


validation_rng = np.random.default_rng(SEED + 1)
validation_times = np.unique(
    np.rint(
        np.linspace(
            valid_start_idx,
            test_start_idx - 1,
            VALIDATION_TIME_COUNT,
        )
    ).astype(np.int64)
)
validation_samples = {}

for time_idx in validation_times:
    eligible_stocks = labeled_stock_indices(int(time_idx))
    if eligible_stocks.size > VALIDATION_STOCKS_PER_TIME:
        eligible_stocks = validation_rng.choice(
            eligible_stocks,
            size=VALIDATION_STOCKS_PER_TIME,
            replace=False,
        )
    validation_samples[int(time_idx)] = np.sort(
        eligible_stocks.astype(np.int64)
    )


def evaluate_sampled_validation():
    rank_ic_values = []
    for time_idx, stock_indices in validation_samples.items():
        if stock_indices.size < 2:
            continue
        predictions = predict_stock_indices(
            time_idx,
            stock_indices,
        )
        labels = data["y1"][time_idx, stock_indices]
        rank_ic_values.append(
            calculate_rank_ic(predictions, labels)
        )
    rank_ic_values = np.asarray(rank_ic_values, dtype=np.float64)
    if not np.any(np.isfinite(rank_ic_values)):
        return np.nan
    return float(np.nanmean(rank_ic_values))


assert sample_training_times(
    np.random.default_rng(SEED)
).size == TRAIN_TIME_BINS * TIMES_PER_BIN
assert len(validation_samples) == VALIDATION_TIME_COUNT

print(
    {
        "sampled_train_times_per_epoch": (
            TRAIN_TIME_BINS * TIMES_PER_BIN
        ),
        "sampled_validation_times": len(validation_samples),
        "sampled_validation_sequences": int(
            sum(
                indices.size
                for indices in validation_samples.values()
            )
        ),
    }
)

{'sampled_train_times_per_epoch': 512, 'sampled_validation_times': 81, 'sampled_validation_sequences': 41472}


## 训练

In [8]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=MAX_EPOCHS,
)
loss_function = nn.MSELoss()
gradient_scaler = torch.amp.GradScaler(
    "cuda",
    enabled=AMP_ENABLED,
)

training_rng = np.random.default_rng(SEED)
training_history = []
best_validation_rank_ic = -np.inf
epochs_without_improvement = 0

for epoch_idx in range(MAX_EPOCHS):
    model.train()
    selected_times = sample_training_times(training_rng)
    epoch_squared_error = 0.0
    epoch_sample_count = 0

    for time_idx in selected_times:
        time_idx = int(time_idx)
        eligible_stocks = labeled_stock_indices(time_idx)
        if eligible_stocks.size == 0:
            continue
        if eligible_stocks.size > STOCKS_PER_TIME:
            eligible_stocks = training_rng.choice(
                eligible_stocks,
                size=STOCKS_PER_TIME,
                replace=False,
            )
        training_rng.shuffle(eligible_stocks)

        for batch_start in range(
            0,
            eligible_stocks.size,
            BATCH_SIZE,
        ):
            batch_stop = min(
                batch_start + BATCH_SIZE,
                eligible_stocks.size,
            )
            batch_stocks = eligible_stocks[
                batch_start:batch_stop
            ]
            batch_inputs, batch_coverage, batch_nonempty = (
                build_model_batch(time_idx, batch_stocks)
            )
            if not np.all(batch_nonempty):
                usable_positions = np.flatnonzero(batch_nonempty)
                batch_stocks = batch_stocks[usable_positions]
                batch_inputs = batch_inputs[usable_positions]
                batch_coverage = batch_coverage[usable_positions]
            if batch_stocks.size == 0:
                continue

            batch_targets = data["y1"][
                time_idx,
                batch_stocks,
            ].astype(np.float32)[:, None]

            input_tensor = torch.from_numpy(
                batch_inputs
            ).to(DEVICE, non_blocking=True)
            coverage_tensor = torch.from_numpy(
                batch_coverage
            ).to(DEVICE, non_blocking=True)
            target_tensor = torch.from_numpy(
                batch_targets
            ).to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(
                device_type=DEVICE.type,
                dtype=torch.float16,
                enabled=AMP_ENABLED,
            ):
                prediction_tensor = model(
                    input_tensor,
                    coverage_tensor,
                )
                loss = loss_function(
                    prediction_tensor,
                    target_tensor,
                )

            gradient_scaler.scale(loss).backward()
            gradient_scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                GRADIENT_CLIP,
            )
            gradient_scaler.step(optimizer)
            gradient_scaler.update()

            batch_count = batch_stocks.size
            epoch_squared_error += float(loss.item()) * batch_count
            epoch_sample_count += batch_count

    current_learning_rate = optimizer.param_groups[0]["lr"]
    assert epoch_sample_count > 0
    scheduler.step()
    epoch_mse = epoch_squared_error / epoch_sample_count
    sampled_validation_rank_ic = evaluate_sampled_validation()

    history_row = {
        "epoch": epoch_idx + 1,
        "learning_rate": current_learning_rate,
        "train_samples": epoch_sample_count,
        "train_mse": epoch_mse,
        "sampled_validation_rank_ic": sampled_validation_rank_ic,
    }
    training_history.append(history_row)
    print(history_row)

    current_score = (
        sampled_validation_rank_ic
        if np.isfinite(sampled_validation_rank_ic)
        else -np.inf
    )
    improved = (
        epoch_idx == 0
        or current_score > best_validation_rank_ic
    )

    if improved:
        best_validation_rank_ic = current_score
        epochs_without_improvement = 0
        torch.save(
            {
                "model_state_dict": {
                    name: value.detach().cpu()
                    for name, value in model.state_dict().items()
                },
                "feature_mean": torch.from_numpy(feature_mean.copy()),
                "feature_std": torch.from_numpy(feature_std.copy()),
                "config": {
                    "window_size": WINDOW_SIZE,
                    "input_channels": MODEL_INPUT_CHANNELS,
                    "hidden_channels": HIDDEN_CHANNELS,
                    "dilations": DILATIONS,
                    "dropout": DROPOUT,
                    "seed": SEED,
                },
                "best_sampled_validation_rank_ic": (
                    sampled_validation_rank_ic
                ),
                "training_history": training_history,
            },
            MODEL_PATH,
        )
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        break

training_history_df = pd.DataFrame(training_history)
display(training_history_df)

{'epoch': 1, 'learning_rate': 0.001, 'train_samples': 102400, 'train_mse': 0.08246948653279106, 'sampled_validation_rank_ic': 0.07975657387847535}
{'epoch': 2, 'learning_rate': 0.0009619397662556434, 'train_samples': 102400, 'train_mse': 0.08196703476074617, 'sampled_validation_rank_ic': 0.0772737229388972}
{'epoch': 3, 'learning_rate': 0.0008535533905932738, 'train_samples': 102400, 'train_mse': 0.08199978271499277, 'sampled_validation_rank_ic': 0.08617161799940957}
{'epoch': 4, 'learning_rate': 0.0006913417161825451, 'train_samples': 102400, 'train_mse': 0.08177679287502543, 'sampled_validation_rank_ic': 0.0803964409515113}
{'epoch': 5, 'learning_rate': 0.0005000000000000001, 'train_samples': 102400, 'train_mse': 0.0814962737186579, 'sampled_validation_rank_ic': 0.087352836298966}
{'epoch': 6, 'learning_rate': 0.0003086582838174552, 'train_samples': 102400, 'train_mse': 0.08127623344917083, 'sampled_validation_rank_ic': 0.08337917942884344}
{'epoch': 7, 'learning_rate': 0.00014644660

,epoch,learning_rate,train_samples,train_mse,sampled_validation_rank_ic
0,1,0.001000,102400,0.082469,0.079757
1,2,0.000962,102400,0.081967,0.077274
2,3,0.000854,102400,0.082000,0.086172
3,4,0.000691,102400,0.081777,0.080396
4,5,0.000500,102400,0.081496,0.087353
5,6,0.000309,102400,0.081276,0.083379
6,7,0.000146,102400,0.080835,0.086414


## 最佳模型恢复

In [9]:
best_checkpoint = torch.load(
    MODEL_PATH,
    map_location=DEVICE,
    weights_only=False,
)
model.load_state_dict(best_checkpoint["model_state_dict"])
model.eval()

print(
    {
        "model_path": str(MODEL_PATH.resolve()),
        "best_sampled_validation_rank_ic": (
            best_checkpoint[
                "best_sampled_validation_rank_ic"
            ]
        ),
    }
)

{'model_path': 'D:\\google_dl\\book\\友安杯\\baseline_outputs\\tcn_y1_model.pt', 'best_sampled_validation_rank_ic': 0.087352836298966}


## 全量验证

In [10]:
full_validation_rank_ic_values = []

for validation_offset, time_idx in enumerate(
    range(valid_start_idx, test_start_idx)
):
    stock_indices = labeled_stock_indices(time_idx)
    if stock_indices.size < 2:
        full_validation_rank_ic_values.append(np.nan)
        continue
    predictions = predict_stock_indices(
        time_idx,
        stock_indices,
    )
    labels = data["y1"][time_idx, stock_indices]
    full_validation_rank_ic_values.append(
        calculate_rank_ic(predictions, labels)
    )
    if (
        validation_offset + 1
    ) % 25 == 0 or time_idx == test_start_idx - 1:
        print(
            f"validation {validation_offset + 1}/"
            f"{test_start_idx - valid_start_idx}"
        )

full_validation_rank_ic_values = np.asarray(
    full_validation_rank_ic_values,
    dtype=np.float64,
)
finite_validation_metrics = np.isfinite(
    full_validation_rank_ic_values
)
assert np.any(finite_validation_metrics)

full_validation_mean_rank_ic = float(
    np.mean(
        full_validation_rank_ic_values[
            finite_validation_metrics
        ]
    )
)
full_validation_std_rank_ic = float(
    np.std(
        full_validation_rank_ic_values[
            finite_validation_metrics
        ]
    )
)

validation_summary = pd.Series(
    {
        "valid_time_points": int(
            finite_validation_metrics.sum()
        ),
        "mean_rank_ic": full_validation_mean_rank_ic,
        "std_rank_ic": full_validation_std_rank_ic,
        "baseline_rank_ic": BASELINE_VALIDATION_RANK_IC,
        "improvement": (
            full_validation_mean_rank_ic
            - BASELINE_VALIDATION_RANK_IC
        ),
    },
    name="full_validation",
)
display(validation_summary)

best_checkpoint["full_validation_mean_rank_ic"] = (
    full_validation_mean_rank_ic
)
best_checkpoint["full_validation_std_rank_ic"] = (
    full_validation_std_rank_ic
)
torch.save(best_checkpoint, MODEL_PATH)

validation 25/243
validation 50/243
validation 75/243
validation 100/243
validation 125/243
validation 150/243
validation 175/243
validation 200/243
validation 225/243
validation 243/243


valid_time_points    243.000000
mean_rank_ic           0.091556
std_rank_ic            0.079583
baseline_rank_ic       0.089678
improvement            0.001878
Name: full_validation, dtype: float64

## 测试集预测

In [11]:
test_time_count = T - test_start_idx
test_predictions = np.full(
    (test_time_count, STOCK_COUNT),
    0.5,
    dtype=np.float32,
)

for output_time_idx, time_idx in enumerate(
    range(test_start_idx, T)
):
    available_stocks = np.flatnonzero(
        history_available_mask(time_idx)
    )
    if available_stocks.size:
        test_predictions[
            output_time_idx,
            available_stocks,
        ] = predict_stock_indices(
            time_idx,
            available_stocks,
        )
    if (
        output_time_idx + 1
    ) % 25 == 0 or time_idx == T - 1:
        print(
            f"test {output_time_idx + 1}/"
            f"{test_time_count}"
        )

np.save(PREDICTION_PATH, test_predictions)

loaded_predictions = np.load(PREDICTION_PATH)
assert loaded_predictions.shape == (
    T - test_start_idx,
    STOCK_COUNT,
)
assert loaded_predictions.dtype == np.float32
assert np.all(np.isfinite(loaded_predictions))

output_summary = pd.Series(
    {
        "path": str(PREDICTION_PATH.resolve()),
        "shape": str(loaded_predictions.shape),
        "dtype": str(loaded_predictions.dtype),
        "minimum": float(loaded_predictions.min()),
        "maximum": float(loaded_predictions.max()),
        "mean": float(loaded_predictions.mean()),
        "neutral_count": int(
            np.count_nonzero(loaded_predictions == 0.5)
        ),
        "file_size_mb": (
            PREDICTION_PATH.stat().st_size / 1024**2
        ),
    },
    name="tcn_y1",
)
display(output_summary)

test 25/442
test 50/442
test 75/442
test 100/442
test 125/442
test 150/442
test 175/442
test 200/442
test 225/442
test 250/442
test 275/442
test 300/442
test 325/442
test 350/442
test 375/442
test 400/442
test 425/442
test 442/442


path             D:\google_dl\book\友安杯\baseline_outputs\tcn_y1.npy
shape                                                  (442, 5282)
dtype                                                      float32
minimum                                                   0.005081
maximum                                                   0.938477
mean                                                      0.511076
neutral_count                                               120524
file_size_mb                                              8.906082
Name: tcn_y1, dtype: object